# Jigsaw SSL pretraining, ps32

This notebook pretrains a modified 14-channel ResNet-18 encoder using a Jigsaw self-supervised learning task on unlabeled conditioning-factor raster patches. The model predicts which predefined 4x4 tile permutation was applied to each patch.


## 1. Purpose and experiment configuration

This notebook only performs Jigsaw SSL pretraining. It does not use landslide/non-landslide labels, does not fine-tune a supervised classifier, and does not generate susceptibility maps.

This is in-domain transductive SSL pretraining: unlabeled patches are sampled from the whole cleaned study area. SCV holdout clusters are not excluded because no landslide/non-landslide labels are used during this pretraining stage.


In [1]:
from pathlib import Path
import os

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

raster_dir = PROJECT_ROOT / "data" / "processed" / "rasters_cleaned"
unlabeled_index_csv = PROJECT_ROOT / "data" / "processed" / "ssl_unlabeled_indices" / "unlabeled_patch_index_ps32_n20000.csv"
permutation_bank_csv = PROJECT_ROOT / "data" / "processed" / "ssl_pretext_configs" / "jigsaw_permutation_bank_ps32_grid4_K100.csv"

output_root = PROJECT_ROOT / "outputs" / "SSL_jigsaw_ps32"
training_log_dir = output_root / "training_logs"
figure_root = PROJECT_ROOT / "figures" / "SSL_jigsaw_ps32"
training_curve_dir = figure_root / "training_curves"
jigsaw_example_dir = figure_root / "jigsaw_examples"
confusion_dir = figure_root / "confusion"
checkpoint_dir = PROJECT_ROOT / "checkpoints" / "ssl_pretrained" / "jigsaw"

training_log_path = training_log_dir / "jigsaw_ps32_training_log.csv"
encoder_best_path = checkpoint_dir / "resnet18_jigsaw_ps32_encoder_best.pt"
full_model_best_path = checkpoint_dir / "resnet18_jigsaw_ps32_full_model_best.pt"
last_checkpoint_path = checkpoint_dir / "resnet18_jigsaw_ps32_last.pt"
loss_acc_curve_png = training_curve_dir / "jigsaw_ps32_loss_accuracy_curve.png"
loss_acc_curve_pdf = training_curve_dir / "jigsaw_ps32_loss_accuracy_curve.pdf"
jigsaw_example_png = jigsaw_example_dir / "jigsaw_examples_ps32.png"
jigsaw_example_pdf = jigsaw_example_dir / "jigsaw_examples_ps32.pdf"
confusion_png = confusion_dir / "jigsaw_val_confusion_top_classes_ps32.png"
confusion_pdf = confusion_dir / "jigsaw_val_confusion_top_classes_ps32.pdf"

patch_size = 32
grid_size = 4
tile_size = patch_size // grid_size
n_tiles = grid_size * grid_size
n_permutation_classes = 100
n_unlabeled_patches = 20_000
normalization_sample_size = 5_000
max_nodata_ratio = 0.0
nodata_value = -9999
random_seed = 42
max_attempts = 1_000_000
train_fraction = 0.9
batch_size = 128
num_workers = 0
learning_rate = 1e-4
weight_decay = 1e-4
max_epochs = 50
early_stopping_patience = 10
gradient_clip_norm = 5.0

print("Project root:", PROJECT_ROOT)
print("Raster dir:", raster_dir)
print("Unlabeled index:", unlabeled_index_csv)
print("Permutation bank:", permutation_bank_csv)
print("Encoder checkpoint output:", encoder_best_path)


Project root: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project
Raster dir: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\data\processed\rasters_cleaned
Unlabeled index: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\data\processed\ssl_unlabeled_indices\unlabeled_patch_index_ps32_n20000.csv
Permutation bank: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\data\processed\ssl_pretext_configs\jigsaw_permutation_bank_ps32_grid4_K100.csv
Encoder checkpoint output: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\jigsaw\resnet18_jigsaw_ps32_encoder_best.pt


## 2. Import packages and project modules


In [2]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset

from src.patch_dataset import list_raster_files, audit_raster_alignment
from src.ssl_masked_recon import MaskedReconstructionRasterDataset, compute_ssl_channel_stats, create_unlabeled_patch_index
from src.ssl_jigsaw import (
    JigsawRasterPatchDataset,
    JigsawResNet18Model,
    apply_jigsaw_permutation,
    load_or_create_jigsaw_permutation_bank,
)
from src.train_ssl import evaluate_jigsaw, train_jigsaw_model
from src.plotting import (
    plot_jigsaw_confusion_top_classes,
    plot_jigsaw_examples,
    plot_jigsaw_training_curves,
    set_publication_plot_style,
)
from src.utils import count_trainable_parameters, ensure_dir, set_global_seed

for directory in [
    unlabeled_index_csv.parent,
    permutation_bank_csv.parent,
    training_log_dir,
    training_curve_dir,
    jigsaw_example_dir,
    confusion_dir,
    checkpoint_dir,
]:
    ensure_dir(directory)


## 3. Set random seeds and device


In [3]:
set_global_seed(random_seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = bool(torch.cuda.is_available())
print("Using device:", device)
if torch.cuda.is_available():
    print("CUDA GPU:", torch.cuda.get_device_name(0))
    print("torch.version.cuda:", torch.version.cuda)
else:
    print("WARNING: CUDA is not available; Jigsaw pretraining will run on CPU.")


Using device: cuda
CUDA GPU: NVIDIA GeForce RTX 4090
torch.version.cuda: 11.8


## 4. Set publication-quality plotting style


In [4]:
plot_font_family = set_publication_plot_style(font_family="Arial", font_size=10)
print("Plotting font family:", plot_font_family)
print("Plotting font size:", 10)


Plotting font family: Arial
Plotting font size: 10


## 5. Load and audit cleaned rasters


In [5]:
raster_files = list_raster_files(raster_dir)
raster_audit = audit_raster_alignment(raster_files, expected_nodata=nodata_value)
print("number of raster files:", len(raster_files))
print("raster shape:", (raster_audit.height, raster_audit.width))
print("raster CRS:", raster_audit.crs)
print("raster nodata:", raster_audit.nodata)
print("patch_size:", patch_size)
print("grid_size:", grid_size)
print("tile_size:", tile_size)
print("n_tiles:", n_tiles)


number of raster files: 14
raster shape: (10071, 9596)
raster CRS: EPSG:32618
raster nodata: -9999.0
patch_size: 32
grid_size: 4
tile_size: 8
n_tiles: 16


## 6. Load or generate unlabeled patch index


In [6]:
unlabeled_index = create_unlabeled_patch_index(
    raster_dir=raster_dir,
    output_csv=unlabeled_index_csv,
    patch_size=patch_size,
    n_patches=n_unlabeled_patches,
    nodata_value=nodata_value,
    max_nodata_ratio=max_nodata_ratio,
    random_seed=random_seed,
    max_attempts=max_attempts,
    force_regenerate=False,
)
valid_unlabeled = unlabeled_index.loc[unlabeled_index["valid_patch"].astype(bool)].copy()
if len(valid_unlabeled) < n_unlabeled_patches:
    raise RuntimeError(f"Expected at least {n_unlabeled_patches} valid unlabeled patches, found {len(valid_unlabeled)}.")
valid_unlabeled = valid_unlabeled.iloc[:n_unlabeled_patches].copy()
if not np.isclose(valid_unlabeled["nodata_ratio"].to_numpy(dtype="float64"), 0.0).all():
    raise ValueError("Unlabeled patch index contains nonzero nodata_ratio rows.")
print("number of valid unlabeled patches:", len(valid_unlabeled))
print(valid_unlabeled[["patch_id", "row", "col", "nodata_ratio"]].head().to_string(index=False))


number of valid unlabeled patches: 20000
    patch_id  row  col  nodata_ratio
U_SSL_000001 2038  916           0.0
U_SSL_000002 5168 1241           0.0
U_SSL_000003 5491 4256           0.0
U_SSL_000004 4538 2189           0.0
U_SSL_000005  940 5320           0.0


## 7. Generate or load Jigsaw permutation bank


In [7]:
permutation_bank = load_or_create_jigsaw_permutation_bank(
    output_csv=permutation_bank_csv,
    n_tiles=n_tiles,
    n_permutations=n_permutation_classes,
    random_seed=random_seed,
    min_hamming_distance=None,
)
if permutation_bank.shape != (n_permutation_classes, n_tiles):
    raise ValueError(f"Unexpected permutation bank shape: {permutation_bank.shape}")
for row in permutation_bank:
    if sorted(row.tolist()) != list(range(n_tiles)):
        raise ValueError("Invalid permutation row found.")
print("Permutation bank shape:", permutation_bank.shape)
print("Permutation bank path:", permutation_bank_csv)
print(pd.read_csv(permutation_bank_csv).head().to_string(index=False))


Permutation bank shape: (100, 16)
Permutation bank path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\data\processed\ssl_pretext_configs\jigsaw_permutation_bank_ps32_grid4_K100.csv
 perm_class  position_00  position_01  position_02  position_03  position_04  position_05  position_06  position_07  position_08  position_09  position_10  position_11  position_12  position_13  position_14  position_15
          0            6           15           11           10            9            3            0            7           12            5            2            4           14            1           13            8
          1           10            8           14            5            4            0            7           15            2            3           12            1           13            6           11            9
          2            4           13            6            9            3            1           10            5            0            2   

## 8. Compute Jigsaw SSL channel normalization statistics


In [8]:
raw_stats_dataset = MaskedReconstructionRasterDataset(
    patch_index_csv=unlabeled_index_csv,
    raster_dir=raster_dir,
    patch_size=patch_size,
    nodata_value=nodata_value,
    normalize=False,
    return_metadata=False,
)
channel_means, channel_stds = compute_ssl_channel_stats(
    raw_stats_dataset,
    sample_size=normalization_sample_size,
    batch_size=64,
    random_seed=random_seed,
)
raw_stats_dataset.close()
print("Channel means shape:", channel_means.shape)
print("Channel stds shape:", channel_stds.shape)
print("Channel means:", np.round(channel_means, 4))
print("Channel stds:", np.round(channel_stds, 4))
if not np.isfinite(channel_means).all() or not np.isfinite(channel_stds).all():
    raise ValueError("Channel normalization statistics contain NaN or inf.")
if (channel_stds <= 0).any():
    raise ValueError("At least one channel std is nonpositive.")


Channel means shape: (14,)
Channel stds shape: (14,)
Channel means: [1.8172540e+02 1.5900377e+03 1.9745800e+01 1.7168539e+02 1.9745800e+01
 4.2806499e+01 1.0002100e+01 6.5240002e-01 1.7899999e-02 1.7500000e-02
 5.2075802e+01 5.5462999e+00 2.4302999e+01 7.0296001e+00]
Channel stds: [7.987370e+01 6.338770e+01 3.289600e+00 1.632223e+02 3.289600e+00
 2.239570e+01 3.565600e+00 1.664000e-01 1.545000e-01 2.632000e-01
 9.396800e+00 5.474800e+00 1.303064e+02 1.296800e+00]


## 9. Define Jigsaw Dataset and DataLoaders


In [9]:
jigsaw_dataset = JigsawRasterPatchDataset(
    patch_index_csv=unlabeled_index_csv,
    raster_dir=raster_dir,
    permutation_bank_csv=permutation_bank_csv,
    patch_size=patch_size,
    grid_size=grid_size,
    nodata_value=nodata_value,
    normalize=True,
    channel_means=channel_means,
    channel_stds=channel_stds,
    return_metadata=False,
    random_seed=random_seed,
)

X_jigsaw, y_perm = jigsaw_dataset[0]
print("Dataset length:", len(jigsaw_dataset))
print("X_jigsaw.shape:", tuple(X_jigsaw.shape))
print("y_perm:", int(y_perm))
print("X_jigsaw finite:", bool(torch.isfinite(X_jigsaw).all()))
print("X_jigsaw contains nodata:", bool((X_jigsaw == nodata_value).any()))
if X_jigsaw.shape != (14, patch_size, patch_size):
    raise ValueError(X_jigsaw.shape)
if not (0 <= int(y_perm) < n_permutation_classes):
    raise ValueError(int(y_perm))
original_patch = jigsaw_dataset.read_normalized_patch(0)
if int(y_perm) != -1 and torch.allclose(original_patch, X_jigsaw):
    print("WARNING: shuffled patch is identical to original; check permutation bank if this happens often.")
else:
    print("Shuffled patch differs from original in smoke test.")

rng = np.random.default_rng(random_seed)
all_indices = np.arange(len(jigsaw_dataset))
rng.shuffle(all_indices)
n_train = int(train_fraction * len(all_indices))
train_indices = all_indices[:n_train].tolist()
val_indices = all_indices[n_train:].tolist()
train_dataset = Subset(jigsaw_dataset, train_indices)
val_dataset = Subset(jigsaw_dataset, val_indices)

loader_generator = torch.Generator()
loader_generator.manual_seed(random_seed)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
    generator=loader_generator,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
)
print("Train/val split sizes:", len(train_dataset), len(val_dataset))
print("batch size:", batch_size)
print("pin_memory:", pin_memory)


Dataset length: 20000
X_jigsaw.shape: (14, 32, 32)
y_perm: 76
X_jigsaw finite: True
X_jigsaw contains nodata: False
Shuffled patch differs from original in smoke test.
Train/val split sizes: 18000 2000
batch size: 128
pin_memory: True


## 10. Define modified ResNet-18 encoder and Jigsaw classification head


In [10]:
model = JigsawResNet18Model(
    in_channels=14,
    n_permutation_classes=n_permutation_classes,
    small_patch_stem=True,
    dropout=0.3,
).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
print("model parameter count:", count_trainable_parameters(model))
print("n_permutation_classes:", n_permutation_classes)


model parameter count: 11226468
n_permutation_classes: 100


## 11. Define Jigsaw training objective and accuracy metrics


In [11]:
smoke_batch = next(iter(train_loader))
smoke_X = smoke_batch[0].to(device, non_blocking=True)
smoke_y = smoke_batch[1].to(device, non_blocking=True)
print("model device:", next(model.parameters()).device)
print("X_jigsaw.device:", smoke_X.device)
print("y_perm.device:", smoke_y.device)
with torch.no_grad():
    smoke_logits = model(smoke_X[:4])
print("logits shape:", tuple(smoke_logits.shape))
if torch.cuda.is_available():
    assert next(model.parameters()).is_cuda and smoke_X.is_cuda and smoke_y.is_cuda


model device: cuda:0
X_jigsaw.device: cuda:0
y_perm.device: cuda:0
logits shape: (4, 100)


## 12. Train Jigsaw SSL model


In [12]:
config = {
    "ssl_task": "jigsaw",
    "patch_size": patch_size,
    "grid_size": grid_size,
    "tile_size": tile_size,
    "n_tiles": n_tiles,
    "n_permutation_classes": n_permutation_classes,
    "n_unlabeled_patches": n_unlabeled_patches,
    "normalization_sample_size": normalization_sample_size,
    "train_fraction": train_fraction,
    "batch_size": batch_size,
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "max_epochs": max_epochs,
    "early_stopping_patience": early_stopping_patience,
    "gradient_clip_norm": gradient_clip_norm,
    "random_seed": random_seed,
    "raster_dir": str(raster_dir),
    "unlabeled_index_csv": str(unlabeled_index_csv),
    "permutation_bank_csv": str(permutation_bank_csv),
}

print("Quality check before training")
print("number of raster files:", len(raster_files))
print("raster shape:", (raster_audit.height, raster_audit.width))
print("patch_size:", patch_size)
print("grid_size:", grid_size)
print("tile_size:", tile_size)
print("n_tiles:", n_tiles)
print("n_permutation_classes:", n_permutation_classes)
print("number of valid unlabeled patches:", len(jigsaw_dataset))
print("train/val split sizes:", len(train_dataset), len(val_dataset))
print("batch size:", batch_size)
print("device:", device)
if torch.cuda.is_available():
    print("CUDA GPU name:", torch.cuda.get_device_name(0))
print("model parameter count:", count_trainable_parameters(model))
print("max_epochs:", max_epochs)
print("learning rate:", learning_rate)
print("encoder checkpoint path:", encoder_best_path)
print("full model checkpoint path:", full_model_best_path)
print("plotting font family and font size:", plot_font_family, 10)

training_log, training_summary = train_jigsaw_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    full_model_best_path=full_model_best_path,
    encoder_best_path=encoder_best_path,
    last_checkpoint_path=last_checkpoint_path,
    config=config,
    channel_means=channel_means,
    channel_stds=channel_stds,
    permutation_bank_path=permutation_bank_csv,
    max_epochs=max_epochs,
    early_stopping_patience=early_stopping_patience,
    batch_size=batch_size,
    n_permutation_classes=n_permutation_classes,
    grad_clip_norm=gradient_clip_norm,
)
training_log.to_csv(training_log_path, index=False)
print("Training summary:", training_summary)
print("Training log saved:", training_log_path)


Quality check before training
number of raster files: 14
raster shape: (10071, 9596)
patch_size: 32
grid_size: 4
tile_size: 8
n_tiles: 16
n_permutation_classes: 100
number of valid unlabeled patches: 20000
train/val split sizes: 18000 2000
batch size: 128
device: cuda
CUDA GPU name: NVIDIA GeForce RTX 4090
model parameter count: 11226468
max_epochs: 50
learning rate: 0.0001
encoder checkpoint path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\jigsaw\resnet18_jigsaw_ps32_encoder_best.pt
full model checkpoint path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\jigsaw\resnet18_jigsaw_ps32_full_model_best.pt
plotting font family and font size: Arial 10


epoch 001: train_loss=4.689856, val_loss=4.630304, val_top1=0.0090, val_top5=0.0500


epoch 002: train_loss=4.648283, val_loss=4.627420, val_top1=0.0115, val_top5=0.0550


epoch 003: train_loss=4.631373, val_loss=4.617215, val_top1=0.0085, val_top5=0.0460


epoch 004: train_loss=4.627668, val_loss=4.609204, val_top1=0.0115, val_top5=0.0560


...

epoch 019: train_loss=4.608774, val_loss=4.609356, val_top1=0.0065, val_top5=0.0420


epoch 020: train_loss=4.608688, val_loss=4.605872, val_top1=0.0085, val_top5=0.0475


epoch 021: train_loss=4.610491, val_loss=4.606462, val_top1=0.0110, val_top5=0.0520


epoch 022: train_loss=4.608078, val_loss=4.608014, val_top1=0.0095, val_top5=0.0540


Training summary: {'best_epoch': 12, 'best_val_loss': 4.605131553649902, 'best_train_loss': 4.6125757649739585, 'best_val_acc_top1': 0.012000000059604644, 'best_val_acc_top5': 0.04650000005960465}
Training log saved: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\SSL_jigsaw_ps32\training_logs\jigsaw_ps32_training_log.csv


## 13. Save best pretrained encoder and logs


In [13]:
required_log_columns = [
    "epoch", "train_loss", "val_loss", "train_acc_top1", "val_acc_top1",
    "train_acc_top5", "val_acc_top5", "learning_rate", "batch_size", "n_permutation_classes",
]
missing_log_columns = [column for column in required_log_columns if column not in training_log.columns]
if missing_log_columns:
    raise ValueError(f"Training log missing columns: {missing_log_columns}")
for path in [training_log_path, encoder_best_path, full_model_best_path, last_checkpoint_path]:
    if not path.exists():
        raise FileNotFoundError(path)
encoder_checkpoint = torch.load(encoder_best_path, map_location="cpu")
full_checkpoint = torch.load(full_model_best_path, map_location="cpu")
required_encoder_keys = {"encoder_state_dict", "epoch", "val_loss", "config", "channel_means", "channel_stds", "permutation_bank_path"}
missing_encoder_keys = required_encoder_keys - set(encoder_checkpoint.keys())
if missing_encoder_keys:
    raise KeyError(f"Encoder checkpoint missing keys: {sorted(missing_encoder_keys)}")
print("Encoder checkpoint keys:", sorted(encoder_checkpoint.keys()))
print("Full model checkpoint keys:", sorted(full_checkpoint.keys()))
print("Encoder checkpoint epoch:", encoder_checkpoint["epoch"])
print("Encoder checkpoint val_loss:", encoder_checkpoint["val_loss"])
print("Encoder state keys:", len(encoder_checkpoint["encoder_state_dict"]))


Encoder checkpoint keys: ['channel_means', 'channel_stds', 'config', 'encoder_state_dict', 'epoch', 'permutation_bank_path', 'val_acc_top1', 'val_acc_top5', 'val_loss']
Full model checkpoint keys: ['channel_means', 'channel_stds', 'config', 'epoch', 'model_state_dict', 'optimizer_state_dict', 'permutation_bank_path', 'train_loss', 'val_acc_top1', 'val_acc_top5', 'val_loss']
Encoder checkpoint epoch: 12
Encoder checkpoint val_loss: 4.605131553649902
Encoder state keys: 120


## 14. Visualize Jigsaw examples and training curves


In [14]:
plot_jigsaw_training_curves(training_log, loss_acc_curve_png)
plot_jigsaw_training_curves(training_log, loss_acc_curve_pdf)

# Build a deterministic visualization example from the first patch and class 0.
example_X = jigsaw_dataset.read_normalized_patch(0).unsqueeze(0)
example_perm_class = 0
example_perm = torch.as_tensor(permutation_bank[example_perm_class], dtype=torch.long)
example_X_jigsaw = apply_jigsaw_permutation(example_X[0], example_perm, grid_size=grid_size).unsqueeze(0)
plot_jigsaw_examples(
    example_X,
    example_X_jigsaw,
    output_path=jigsaw_example_png,
    perm_class=example_perm_class,
    grid_size=grid_size,
    channel_indices=[0, 1, 2],
    channel_names=["factor_01", "factor_02", "factor_03"],
)
plot_jigsaw_examples(
    example_X,
    example_X_jigsaw,
    output_path=jigsaw_example_pdf,
    perm_class=example_perm_class,
    grid_size=grid_size,
    channel_indices=[0, 1, 2],
    channel_names=["factor_01", "factor_02", "factor_03"],
)

# Diagnostic only: selected-class validation confusion matrix.
print("NOTE: Jigsaw validation confusion is diagnostic only and is not a landslide susceptibility metric.")
best_full = torch.load(full_model_best_path, map_location=device)
model.load_state_dict(best_full["model_state_dict"])
val_pred_result = evaluate_jigsaw(model, val_loader, criterion, device, return_predictions=True)
plot_jigsaw_confusion_top_classes(val_pred_result["y_true"], val_pred_result["y_pred"], confusion_png, n_classes_to_show=20)
plot_jigsaw_confusion_top_classes(val_pred_result["y_true"], val_pred_result["y_pred"], confusion_pdf, n_classes_to_show=20)

for path in [loss_acc_curve_png, loss_acc_curve_pdf, jigsaw_example_png, jigsaw_example_pdf, confusion_png, confusion_pdf]:
    if not path.exists():
        raise FileNotFoundError(path)
    print("Saved figure:", path)


NOTE: Jigsaw validation confusion is diagnostic only and is not a landslide susceptibility metric.


Saved figure: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\SSL_jigsaw_ps32\training_curves\jigsaw_ps32_loss_accuracy_curve.png
Saved figure: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\SSL_jigsaw_ps32\training_curves\jigsaw_ps32_loss_accuracy_curve.pdf
Saved figure: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\SSL_jigsaw_ps32\jigsaw_examples\jigsaw_examples_ps32.png
Saved figure: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\SSL_jigsaw_ps32\jigsaw_examples\jigsaw_examples_ps32.pdf
Saved figure: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\SSL_jigsaw_ps32\confusion\jigsaw_val_confusion_top_classes_ps32.png
Saved figure: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\SSL_jigsaw_ps32\confusion\jigsaw_val_confusion_top_classes_ps32.pdf


## 15. Print final summary and next-step notes


In [15]:
final_train_loss = float(training_log["train_loss"].iloc[-1])
final_val_loss = float(training_log["val_loss"].iloc[-1])
print("Jigsaw SSL pretraining summary")
print("number of valid unlabeled patches:", len(jigsaw_dataset))
print("best epoch:", training_summary["best_epoch"])
print("best validation loss:", training_summary["best_val_loss"])
print("best validation top-1 accuracy:", training_summary["best_val_acc_top1"])
print("best validation top-5 accuracy:", training_summary["best_val_acc_top5"])
print("final train loss:", final_train_loss)
print("final val loss:", final_val_loss)
print("encoder checkpoint path:", encoder_best_path)
print("full model checkpoint path:", full_model_best_path)
print("last checkpoint path:", last_checkpoint_path)
print("training log path:", training_log_path)
print("training curve figure path:", loss_acc_curve_png)
print("Jigsaw example figure path:", jigsaw_example_png)
print("confusion figure path:", confusion_png)
print("Confirmed: no landslide labels were used, no supervised fine-tuning was performed, and no susceptibility maps were generated.")
print("Next step: use the Jigsaw encoder checkpoint in supervised SCV fine-tuning notebook: notebooks/12_finetune_jigsaw_resnet18_ps32_scv.ipynb")
jigsaw_dataset.close()


Jigsaw SSL pretraining summary
number of valid unlabeled patches: 20000
best epoch: 12
best validation loss: 4.605131553649902
best validation top-1 accuracy: 0.012000000059604644
best validation top-5 accuracy: 0.04650000005960465
final train loss: 4.608078384823269
final val loss: 4.60801354598999
encoder checkpoint path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\jigsaw\resnet18_jigsaw_ps32_encoder_best.pt
full model checkpoint path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\jigsaw\resnet18_jigsaw_ps32_full_model_best.pt
last checkpoint path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\jigsaw\resnet18_jigsaw_ps32_last.pt
training log path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\SSL_jigsaw_ps32\training_logs\jigsaw_ps32_training_log.csv
training curve figure path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\fi